# Same loop, different brain

**Block 2, How It Actually Works**

In Block 1 you saw an agent loop that drove Claude through a small
debugging task. The slides for this block argued that **post-training is
convergent**, every major lab uses some version of `pre-train -> SFT
-> preference learning -> tool-use FT`, which is why models trained by
different companies all *behave like agents*.

This notebook makes that claim tactile. We are going to:

1. Import the **same agent loop** Block 1 walked through (now packaged
   for reuse in `workshop_agent`).
2. Discover which models the workshop's LiteLLM proxy actually fronts.
3. Run the loop against each available model on the **same task**.
4. Compare traces.

Then a bonus experiment on the prompt-side lever (tool descriptions).

## 1. Setup

Same setup as Block 1's notebook. The new line is `from workshop_agent
import ...`, the agent loop, the tool handlers, the schemas, all of it
lives in a tiny package now (`workshop_agent/src/workshop_agent/agent.py`).
We freed up this notebook to focus on what is new.

In [ ]:
import os
from pathlib import Path

import litellm

from workshop_agent import TOOLS, Sandbox, make_tool_handlers, run_agent

# Load a local .env if one exists (no-op in the Codespace, where the
# gateway credentials arrive as injected secrets). load_dotenv does NOT
# override variables already in the environment, so secrets always win.
try:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
except ModuleNotFoundError:
    pass

# The Codespace injects the gateway URL as LITELLM_BASE_URL (see
# .devcontainer/devcontainer.json); a local .env file can set the same
# variable when running outside the Codespace.
litellm.api_base = os.environ.get("LITELLM_BASE_URL")
litellm.api_key = os.environ.get("LITELLM_API_KEY")

if not litellm.api_base or not litellm.api_key:
    raise RuntimeError(
        "Missing gateway credentials. Set LITELLM_BASE_URL "
        "and LITELLM_API_KEY as Codespace secrets, export them in your shell, or "
        "put them in a .env file at the repo root."
    )

# Reuse Block 1's starter project as the sandbox: no need to copy it.
SANDBOX_ROOT = (Path.cwd() / ".." / ".." / "01-landscape" / "demo" / "starter").resolve()
sandbox = Sandbox(SANDBOX_ROOT)
TOOL_HANDLERS = make_tool_handlers(sandbox)

print("api_base:", litellm.api_base)
print("sandbox: ", sandbox.root)

## 2. Reset Block 1's starter

Whether or not Block 1's demo ran, we need `converters.py` in its known
buggy state so each model sees the same task.

In [ ]:
BUGGY = '''"""Temperature unit conversions.

The Fahrenheit/Celsius converter contains a deliberate bug used by the
Block 1 demo. Do not "fix" it outside the demo flow.
"""


def fahrenheit_to_celsius(f: float) -> float:
    """Convert a temperature in degrees Fahrenheit to degrees Celsius."""
    return (f - 32) * 9 / 5


def celsius_to_fahrenheit(c: float) -> float:
    """Convert a temperature in degrees Celsius to degrees Fahrenheit."""
    return c * 9 / 5 + 32


def kelvin_to_celsius(k: float) -> float:
    """Convert a temperature in Kelvin to degrees Celsius."""
    return k - 273.15
'''

(sandbox.root / "src" / "sci_units" / "converters.py").write_text(BUGGY)
print("reset converters.py to buggy starting state")

## 3. What models does the proxy actually front?

We do not know in advance whether the LLM proxy server fronts only
Claude, or also GPT and Gemini, or something else entirely. So we
*ask*, by sending a one-token ping to each candidate alias and
keeping the ones that answer.

This is a useful pattern in its own right: the same notebook will keep
working as the proxy admin adds or removes models.

In [ ]:
CANDIDATE_MODELS = [
    "claude-sonnet-4-6",
    "claude-haiku-4-5",
    # "gpt-5",
    # "gpt-5-mini",
    # "gemini-2.5-pro",
    # "gemini-2.5-flash",
]


def model_is_available(model: str) -> bool:
    try:
        litellm.completion(
            model=model,
            messages=[{"role": "user", "content": "ping"}],
            max_tokens=4,
        )
        return True
    except Exception as e:
        print(f"  {model}: unavailable ({type(e).__name__})")
        return False


AVAILABLE = [m for m in CANDIDATE_MODELS if model_is_available(m)]
print()
print("available on this proxy:")
for m in AVAILABLE:
    print(f"  - {m}")

if not AVAILABLE:
    raise RuntimeError(
        "No candidate models reachable. Check LITELLM_API_KEY / LITELLM_BASE_URL, "
        "or update CANDIDATE_MODELS to match the aliases your proxy exposes."
    )

## 4. The agent loop, in one line

Block 1's notebook spent five cells building the agent loop from
scratch. We are going to reuse it now via a single import. Read the
signature: `run_agent` is the same function you walked through in
Block 1, parameterized so we can pass a different `model=` each time.

In [ ]:
import inspect

print(inspect.signature(run_agent))

## 5. A small driver

To compare models cleanly we want **the same starting messages, the same
tools, the same task, fresh state**. This wrapper:

1. Resets `converters.py` (so each model sees the same broken file).
2. Builds a fresh message list (so models cannot peek at each other's traces).
3. Calls `run_agent` with the supplied `model=`.
4. Catches errors so one model failing does not kill the demo.

In [ ]:
SYSTEM_PROMPT = """You are a careful coding agent working inside a small Python project.

You have three tools: read_file, write_file, run_bash. All paths are relative
to the project root (the directory containing pyproject.toml).

Your job: investigate problems, propose minimal fixes, and verify by running
the test suite. Keep changes small. Always re-run tests after editing.

When you are done, reply with a short plain-text summary of what you did.
"""

USER_PROMPT = (
    "There is a failing test in this project. "
    "Find it, understand the bug, fix it, and verify with pytest. "
    "Use the smallest possible change."
)


def compare(model: str, *, max_steps: int = 10) -> None:
    print(f"\n{'=' * 70}\nMODEL: {model}\n{'=' * 70}")
    # reset the file so each model sees the same starting state
    (sandbox.root / "src" / "sci_units" / "converters.py").write_text(BUGGY)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]
    try:
        run_agent(
            model=model,
            messages=messages,
            tool_handlers=TOOL_HANDLERS,
            tools=TOOLS,
            max_steps=max_steps,
        )
    except Exception as e:
        print(f"\n=== {model} raised {type(e).__name__}: {e} ===")

## 6. Run it against each available model

Watch each trace. The interesting comparisons:

- Which tool does the model call **first**? (`run_bash pytest`? Or
  `read_file converters.py`?)
- How many turns does it take?
- Does it write the same fix? Does it explain the fix?
- Does any model **fail** in an interesting way? (Failure is data, it
  feeds Block 3.)

In [ ]:
for model in AVAILABLE[:3]:  # cap at 3 for time
    compare(model)

## 7. What we just saw

Same loop. Same tools. Same task. Different brains.

If post-training were *not* convergent, if Anthropic, OpenAI, and
Google had taken radically different approaches, we would expect at
least one of these models to refuse, to flail, or to misuse the tools.
Instead, they all (mostly) drove the loop to a passing test, with
small stylistic differences.

> **Convergence is not perfection.** Some models will be faster,
> some will write a more elegant fix, some will explain themselves more
> verbosely. But the *shape* of the behavior is the same, and that
> shape is what `pre-train -> SFT -> RLHF -> tool-use FT` produces
> across labs. The tactile proof is right above this cell.

## 8. Bonus: prompt-side levers are real

The slides framed every agent surprise as either a **training problem**
or a **prompt problem**, with the prompt being the part you control. Let
us make that lever tangible, *deliberately* sabotage the prompt and watch
behavior degrade.

A first attempt at sabotage, blanking out the *descriptions*, barely
dents a modern model. That is the lesson hiding in plain sight: the real
signal a tool schema carries is not the prose, it is the **names**.
`read_file(path)`, `write_file(path, content)`, `run_bash(command)` are
self-documenting. Haiku and Sonnet reverse-engineer the whole API from
the names alone and sail through.

So below we strip the names too. `VAGUE_TOOLS` renames the tools to
opaque labels (`alpha`, `beta`, `gamma`), renames every parameter
(`x1`, `x2`), and gives each a useless or misleading one-liner. The
underlying handlers are unchanged, we just hide what they *are* from the
model. Same model. Same task. Watch the loop flail.

(If your model still somehow solves it, make it worse: shuffle the
descriptions so they point at the wrong tool.)

In [ ]:
import copy

# To genuinely sabotage the schema we hide the THREE things a model reads
# off a tool: its name, its parameter names, and its description. We map
# each real tool to an opaque alias with renamed params and a useless or
# misleading one-liner. The handlers underneath never change, so any
# flailing you see is the MODEL degrading on a bad prompt, not the harness
# crashing.
OBFUSCATION = {
    "read_file": {
        "alias": "alpha",
        "description": "Does something. Use only when necessary.",
        "params": {"path": "x1"},  # real -> obfuscated
    },
    "write_file": {
        "alias": "beta",
        "description": "A tool.",
        "params": {"path": "x1", "content": "x2"},
    },
    "run_bash": {
        "alias": "gamma",
        "description": "Returns a value.",
        "params": {"command": "x1"},
    },
}

VAGUE_TOOLS = []
for t in copy.deepcopy(TOOLS):
    fn = t["function"]
    spec = OBFUSCATION[fn["name"]]
    fn["name"] = spec["alias"]
    fn["description"] = spec["description"]
    props = fn["parameters"]["properties"]
    new_props = {}
    for real_param, obf_param in spec["params"].items():
        prop = props[real_param]
        prop.pop("description", None)  # blank the param doc too
        new_props[obf_param] = prop
    fn["parameters"]["properties"] = new_props
    fn["parameters"]["required"] = list(spec["params"].values())
    VAGUE_TOOLS.append(t)


def _make_vague_handler(real_name: str, params: dict[str, str]):
    """Wrap a real handler so it accepts the obfuscated param names."""
    real_handler = TOOL_HANDLERS[real_name]
    obf_to_real = {obf: real for real, obf in params.items()}

    def handler(**kwargs):
        return real_handler(**{obf_to_real[k]: v for k, v in kwargs.items()})

    return handler


VAGUE_TOOL_HANDLERS = {
    spec["alias"]: _make_vague_handler(real_name, spec["params"])
    for real_name, spec in OBFUSCATION.items()
}


def run_with_tools(model: str, tools_to_use: list, tool_handlers: dict = TOOL_HANDLERS) -> None:
    print(f"\n--- {model} with {'verbose' if tools_to_use is TOOLS else 'vague'} tools ---")
    (sandbox.root / "src" / "sci_units" / "converters.py").write_text(BUGGY)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]
    try:
        run_agent(
            model=model,
            messages=messages,
            tool_handlers=tool_handlers,
            tools=tools_to_use,
            max_steps=10,
        )
    except Exception as e:
        print(f"=== {model} raised {type(e).__name__}: {e} ===")


# Pick one model and compare the two prompt variants
demo_model = AVAILABLE[0]
run_with_tools(demo_model, TOOLS)
run_with_tools(demo_model, VAGUE_TOOLS, VAGUE_TOOL_HANDLERS)

## 9. Bridge to Block 3

You may have noticed something interesting: not every model behaves
identically, and not every prompt variation lands cleanly. Some models
might have looped. Some might have written extra files. Some might have
hallucinated a function name that does not exist in `sci_units`.

That is **data, not bugs**. Each of those failures has a name and a
mitigation, and they map back to the post-training pipeline you just
learned. *(Looping? Probably trained on short trajectories. Hallucinating
a function? Knowledge cutoff or pattern bias. Refusing? RLHF over-tuned.)*